## Interactive Entity Mapping for PII Evaluation

This notebook shows how to use `CanonicalMapper` to align entity labels between
your dataset and the model being evaluated. Proper entity mapping is essential
for accurate evaluation metrics.

**Topics covered:**
1. Why entity mapping is needed
2. Auto-resolving labels with `CanonicalMapper`
3. Reviewing the resolution output
4. What mapping to `None` means (and its impact on evaluation)
5. Manually resolving pending labels
6. How the final mapping dict is used in evaluation

**Prerequisites:** Basic familiarity with Presidio Evaluator concepts from notebooks 4 and 5.

In [1]:
from collections import Counter
from pathlib import Path
from pprint import pprint

from presidio_analyzer import AnalyzerEngine

from presidio_evaluator import InputSample
from presidio_evaluator.entity_mapping import CanonicalMapper
from presidio_evaluator.evaluation import SpanEvaluator
from presidio_evaluator.models import PresidioAnalyzerWrapper

## 1. Why Entity Mapping is Needed

When evaluating a PII detection model, you're comparing:
- **Dataset entities**: The ground truth labels in your evaluation dataset (e.g., `STREET_ADDRESS`, `FIRST_NAME`, `GPE`)
- **Model entities**: The entity types the model can detect (e.g., `LOCATION`, `PERSON`, `NRP`)

These often use **different naming conventions**:

| Dataset Entity | Model Entity | Notes |
|---------------|--------------|-------|
| `STREET_ADDRESS` | `LOCATION` | Different granularity |
| `GPE` (geopolitical entity) | `LOCATION` | Geographic entities |
| `FIRST_NAME`, `LAST_NAME` | `PERSON` | Model groups all names |
| `IP_ADDRESS` | `IP_ADDRESS` | Exact match ✓ |
| `INTERNAL_CODE` | ❌ | Model doesn't support this |

Without proper mapping, the evaluator would count correct predictions as errors because the entity names don't match.

The solution is to map both sets of entities to a canonical representation: a shared taxonomy where `STREET_ADDRESS`, `GPE`, and `LOCATION` all resolve to the same canonical entity. `CanonicalMapper` does this automatically using exact alias matching, country-prefix patterns, and fuzzy string similarity, so you only need to intervene for labels it cannot resolve on its own.

In [3]:
# Load dataset
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))
print(f"Loaded {len(dataset)} samples")

# Show dataset entities
dataset_entities = Counter()
for sample in dataset:
    for span in sample.spans:
        dataset_entities[span.entity_type] += 1

print(f"\nDataset has {len(dataset_entities)} entity types:")
pprint(dict(dataset_entities.most_common(10)), compact=True)

tokenizing input:   0%|          | 0/1500 [00:00<?, ?it/s]

loading model en_core_web_sm


tokenizing input: 100%|██████████| 1500/1500 [00:04<00:00, 337.86it/s]

Loaded 1500 samples

Dataset has 17 entity types:
{'AGE': 74,
 'CREDIT_CARD': 136,
 'DATE_TIME': 119,
 'GPE': 411,
 'NRP': 55,
 'ORGANIZATION': 250,
 'PERSON': 857,
 'PHONE_NUMBER': 92,
 'STREET_ADDRESS': 598,
 'TITLE': 92}


In [4]:
# Load Presidio Analyzer
analyzer = AnalyzerEngine(default_score_threshold=0.4)

print(f"Model supports {len(analyzer.get_supported_entities('en'))} entity types:")
pprint(sorted(analyzer.get_supported_entities("en")), compact=True)

Model supports 19 entity types:
['CREDIT_CARD', 'CRYPTO', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE',
 'IP_ADDRESS', 'LOCATION', 'MAC_ADDRESS', 'MEDICAL_LICENSE', 'NRP', 'PERSON',
 'PHONE_NUMBER', 'UK_NHS', 'URL', 'US_BANK_NUMBER', 'US_DRIVER_LICENSE',
 'US_ITIN', 'US_PASSPORT', 'US_SSN']


## 2. Auto-Resolving Labels with `CanonicalMapper`

`CanonicalMapper` takes the set of raw entity labels from your dataset and resolves
each one to a canonical entity through a tiered matching strategy:

1. **EXACT** — label is already a canonical name or a known alias
2. **COUNTRY** — label has a country prefix (e.g. `GERMANY_PASSPORT_NUMBER` → `PASSPORT`)
3. **FUZZY** — close string match to a known alias (≥ 80% similarity by default)
4. **PENDING** — no automatic match found; requires manual resolution via `.map()` or `.resolve_interactively()`

Use `from_dataset()` to extract labels automatically from a list of `InputSample` objects.

In [5]:
# Build mapper from dataset — auto-resolves all labels it can
mapper = CanonicalMapper.from_dataset(dataset)

if isinstance(mapper, dict):
    print("All labels resolved automatically. Mapping:", mapper)
else:
    # Rich HTML audit table showing tier (EXACT / COUNTRY / FUZZY / PENDING) per label
    mapper.render_html()

[UNRESOLVED] GPE  — no automatic match found


Raw Label,Tier,Canonical,Score
GPE,PENDING,pending,—
DOMAIN_NAME,FUZZY,NAME,80%
US_DRIVER_LICENSE,EXACT,DRIVER_LICENSE,—
PERSON,EXACT,PERSON,—
IBAN_CODE,EXACT,FINANCIAL,—
TITLE,EXACT,TITLE,—
EMAIL_ADDRESS,EXACT,EMAIL_ADDRESS,—
ORGANIZATION,EXACT,ORGANIZATION,—
DATE_TIME,EXACT,DATE_TIME,—
CREDIT_CARD,EXACT,FINANCIAL,—


## 3. Understanding the Resolution Output

The `render_html()` table shows one row per label with four columns:

| Column | Meaning |
|--------|---------|
| **Raw Label** | The entity type string from your dataset |
| **Tier** | How it was resolved: `EXACT`, `COUNTRY`, `FUZZY`, `COUNTRY_FALLBACK`, `MANUAL`, `NONE`, or `PENDING` |
| **Canonical** | The resolved canonical entity name, or `None` if suppressed |
| **Score** | Similarity score for FUZZY matches; `—` otherwise |

### Tier colour coding
| Tier | Colour | Meaning |
|------|--------|---------|
| EXACT / COUNTRY / MANUAL | Green / Blue | High-confidence resolution |
| FUZZY | Yellow | Good match but worth reviewing |
| COUNTRY_FALLBACK | Amber | Country prefix matched but suffix unknown → defaulted to `NATIONAL_ID` |
| NONE | Grey | Label suppressed (mapped to `None`) |
| PENDING | Red | No automatic match — action required |

In [ ]:
# Plain-text version (useful in scripts / CI logs)
if not isinstance(mapper, dict):
    mapper._print_text()


MAPPING: 14 mapped, 3 unmapped | Dataset: 17 active, 0 excluded | Model: 19 entities

⚠️  UNMAPPED (3):
  • AGE (74 samples)
  • ORGANIZATION (199 samples)
  • TITLE (92 samples)
  💡 You can: 1) map to a model entity, 2) map to None (FN penalty), or 3) exclude them.


## 4. What Does Mapping to `None` Mean?

When you map a dataset entity to `None`, you're telling the evaluator:

> "This entity exists in my dataset, but my model **cannot detect it**.  
> I want to **keep it in evaluation** and **penalize the model** for missing it."

### Mapping to `None` vs Excluding

| Action | Effect on Evaluation |
|--------|---------------------|
| `mapper.map({"ENTITY": None})` | Entity stays in dataset. All instances become **False Negatives** (FN). This lowers recall. |
| Filter the dataset before creating the mapper | Entity is **removed** from evaluation entirely — not counted in any metrics. |

### When to Use Each

**Map to `None`** when:
- The entity is real PII your model *should* detect but doesn't
- You want to capture the true recall (penalise the model for missing it)
- Example: Model doesn't support `AGE`, but age is PII you care about

**Exclude** when:
- The entity is irrelevant to your use case or is noise
- Example: `DEBUG_INFO` or `INTERNAL_CODE` that isn't real PII

In [ ]:
# Example: suppress TITLE — "Mr.", "Dr." are dataset labels the model doesn't detect.
# Mapping to None keeps them in evaluation as False Negatives.
if not isinstance(mapper, dict):
    mapper.map({"TITLE": None})
    mapper.render_html()

✓ Mapping set: TITLE → None
   (2 entities still unmapped)

MAPPING: 14 mapped, 2 unmapped | Dataset: 17 active, 0 excluded | Model: 19 entities

⚠️  UNMAPPED (2):
  • AGE (74 samples)
  • ORGANIZATION (199 samples)
  💡 You can: 1) map to a model entity, 2) map to None (FN penalty), or 3) exclude them.

🔧 Manual (1): TITLE→None


In [ ]:
# Resolve remaining pending labels manually.
# Provide a dict of {raw_label: canonical_entity} — or None to suppress.
# Check mapper.pending to see which labels still need attention.
if not isinstance(mapper, dict):
    print("Pending labels:", mapper.pending)

    # Example: map custom/unknown labels
    # mapper.map({
    #     "MY_CUSTOM_LABEL": "PERSON",      # map to existing canonical
    #     "INTERNAL_CODE": None,             # suppress from evaluation
    # })

    mapper.render_html()

✓ Mapping set: AGE → DATE_TIME
   (1 entities still unmapped)
✓ Mapping set: ORGANIZATION → None


Dataset Entity,→ Model Entity,Samples,Confidence
○ ORGANIZATION,None,199,None ✎
○ TITLE,None,92,None ✎
✓ CREDIT_CARD,CREDIT_CARD,136,1.00
✓ DATE_TIME,DATE_TIME,119,1.00
✓ DOMAIN_NAME,URL,37,1.00
✓ EMAIL_ADDRESS,EMAIL_ADDRESS,49,1.00
✓ GPE,LOCATION,325,1.00
✓ IBAN_CODE,IBAN_CODE,21,1.00
✓ IP_ADDRESS,IP_ADDRESS,14,1.00
✓ PERSON,PERSON,637,1.00


### Impact of Mapping to `None` on Metrics

Labels mapped to `None` stay in the dataset but the model can never predict them correctly,
so every instance becomes a False Negative. This accurately captures the model's real-world
recall for those entity types.

In [ ]:
# Count instances affected by None mappings
if not isinstance(mapper, dict):
    mapping_so_far = {label: rec.canonical for label, rec in mapper._records.items()}
    none_mapped = [
        label for label, canonical in mapping_so_far.items() if canonical is None
    ]

    affected = sum(
        1
        for sample in dataset
        for span in sample.spans
        if span.entity_type in none_mapped
    )
    print(f"Labels mapped to None: {none_mapped}")
    print(
        f"Dataset spans affected (will become FN): {affected} of {sum(len(s.spans) for s in dataset)}"
    )

Entities mapped to None (will become False Negatives):
  • ORGANIZATION: 199 samples → all will be FN
  • TITLE: 92 samples → all will be FN

Total False Negatives from None mappings: 291


## 5. Interactive Resolution for Pending Labels

When there are still pending labels after auto-resolution and `.map()`, you can
call `.resolve_interactively()` to handle them one at a time in the terminal.
It shows ranked fuzzy suggestions; you can pick by number, type a canonical name,
or type `NONE` to suppress.

For notebooks, it's often clearer to just use `.map({...})` directly once you've
decided what each pending label should map to.

In [ ]:
# Customising the hierarchy: pass a custom EntityHierarchy to handle edge cases
# or add aliases not in the default taxonomy.
from presidio_evaluator.entity_mapping import EntityHierarchy

custom_h = EntityHierarchy.default().copy()
custom_h.add_alias("PERSON", "HUMAN_NAME")  # make HUMAN_NAME resolve to PERSON
custom_h.add_alias("EMAIL_ADDRESS", "E_MAIL")

# Build a mapper that uses your custom hierarchy
custom_mapper = CanonicalMapper.from_dataset(dataset, hierarchy=custom_h)
if not isinstance(custom_mapper, dict):
    custom_mapper.render_html()
else:
    print("All resolved:", custom_mapper)

Semantic similarity examples:
  FIRST_NAME → DATE_TIME
  STREET_ADDRESS → EMAIL_ADDRESS
  SSN → US_SSN
  BIRTHDAY → PERSON


In [ ]:
# Adjusting the fuzzy threshold: lower = more aggressive auto-resolution
# (may match labels that shouldn't be grouped together)
loose_mapper = CanonicalMapper.from_dataset(dataset, fuzzy_threshold=0.65)
if not isinstance(loose_mapper, dict):
    print(f"Pending with threshold=0.65: {loose_mapper.pending}")
    loose_mapper.render_html()
else:
    print("All resolved:", loose_mapper)

Mapping using SemanticEntityMapper:

MAPPING: 13 mapped, 4 unmapped | Dataset: 17 active, 0 excluded | Model: 19 entities

⚠️  UNMAPPED (4):
  • AGE (74 samples)
  • GPE (325 samples)
  • ORGANIZATION (199 samples)
  • TITLE (92 samples)
  💡 You can: 1) map to a model entity, 2) map to None (FN penalty), or 3) exclude them.


In [ ]:
# Use canonical_depth to change what counts as a "canonical" entity.
# Default depth=3: e.g. CREDIT_CARD rolls up to FINANCIAL (depth-3).
# depth=4 would keep CARD_NUMBER as canonical instead.
deep_mapper = CanonicalMapper.from_dataset(dataset, canonical_depth=4)
if not isinstance(deep_mapper, dict):
    print(f"Pending with canonical_depth=4: {deep_mapper.pending}")
else:
    from pprint import pprint

    pprint(deep_mapper, compact=True)

With strict threshold (0.7), fewer entities are auto-mapped:

MAPPING: 14 mapped, 3 unmapped | Dataset: 17 active, 0 excluded | Model: 19 entities

⚠️  UNMAPPED (3):
  • AGE (74 samples)
  • ORGANIZATION (199 samples)
  • TITLE (92 samples)
  💡 You can: 1) map to a model entity, 2) map to None (FN penalty), or 3) exclude them.


In [ ]:
# Inspect pending labels and their closest suggestions programmatically
import difflib

from presidio_evaluator.entity_mapping import ALL_CANONICAL_ENTITIES

if not isinstance(mapper, dict) and mapper.pending:
    print("Labels still pending:")
    for label in mapper.pending:
        norm = label.upper().replace("_", "").replace("-", "")
        suggestions = difflib.get_close_matches(
            norm,
            [c.upper().replace("_", "") for c in ALL_CANONICAL_ENTITIES],
            n=3,
            cutoff=0.4,
        )
        print(f"  {label!r:30s}  suggestions: {suggestions}")

Programmatic mapping with confidence scores:
  AGE                  → None                 (confidence: 0.00)
  CREDIT_CARD          → CREDIT_CARD          (confidence: 1.00)
  DATE_TIME            → DATE_TIME            (confidence: 1.00)
  DOMAIN_NAME          → URL                  (confidence: 1.00)
  EMAIL_ADDRESS        → EMAIL_ADDRESS        (confidence: 1.00)
  GPE                  → LOCATION             (confidence: 1.00)
  IBAN_CODE            → IBAN_CODE            (confidence: 1.00)
  IP_ADDRESS           → IP_ADDRESS           (confidence: 1.00)
  NRP                  → NRP                  (confidence: 1.00)
  ORGANIZATION         → None                 (confidence: 0.00)
  PERSON               → PERSON               (confidence: 1.00)
  PHONE_NUMBER         → PHONE_NUMBER         (confidence: 1.00)
  STREET_ADDRESS       → LOCATION             (confidence: 1.00)
  TITLE                → None                 (confidence: 0.00)
  US_DRIVER_LICENSE    → US_DRIVER_LICENSE   

## 6. Getting the Final Mapping and Running Evaluation

Once all labels are resolved (no `pending`), call `.get_mapping()` to get the
`{raw_label: canonical_entity | None}` dict that `SpanEvaluator` expects.

`entities_to_keep` controls **which model predictions count** during evaluation:
- Pass the set of canonical entities from the mapping to ignore model predictions
  for entity types not present in your dataset (avoids spurious False Positives).
- Pass `None` to evaluate against all model entity types.

In [ ]:
# Ensure all pending labels are handled before calling get_mapping()
if not isinstance(mapper, dict):
    if mapper.pending:
        print(f"Still {len(mapper.pending)} pending: {mapper.pending}")
        print("Call mapper.map({...}) or mapper.resolve_interactively() first.")
    else:
        entity_mapping = mapper.get_mapping()
        print("Entity mapping ready:")
        pprint(entity_mapping, compact=True)
else:
    entity_mapping = mapper
    print("Entity mapping (all auto-resolved):")
    pprint(entity_mapping, compact=True)

All entities are already mapped!


In [ ]:
# entities_to_keep: the set of canonical entities we actually want to evaluate.
# This prevents FPs from model entity types not present in the dataset.
model_entities_to_use = {v for v in entity_mapping.values() if v is not None}

print(f"Entity mapping ({len(entity_mapping)} labels):")
pprint(entity_mapping, compact=True)
print(f"\nModel entities to evaluate ({len(model_entities_to_use)}):")
pprint(sorted(model_entities_to_use), compact=True)

Entity Mapping (dataset → model):
{'AGE': 'DATE_TIME',
 'CREDIT_CARD': 'CREDIT_CARD',
 'DATE_TIME': 'DATE_TIME',
 'DOMAIN_NAME': 'URL',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'GPE': 'LOCATION',
 'IBAN_CODE': 'IBAN_CODE',
 'IP_ADDRESS': 'IP_ADDRESS',
 'NRP': 'NRP',
 'PERSON': 'PERSON',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'STREET_ADDRESS': 'LOCATION',
 'US_DRIVER_LICENSE': 'US_DRIVER_LICENSE',
 'US_SSN': 'US_SSN',
 'ZIP_CODE': 'LOCATION'}

Model entities to use (14):
['CREDIT_CARD', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE', 'IP_ADDRESS',
 'LOCATION', 'NRP', 'ORGANIZATION', 'PERSON', 'PHONE_NUMBER', 'TITLE', 'URL',
 'US_DRIVER_LICENSE', 'US_SSN']


In [ ]:
# Compare evaluation strategies
all_model_entities = set(analyzer.get_supported_entities("en"))
filtered_entities = set(model_entities_to_use)

print("=== Evaluation Strategy Comparison ===")
print(f"\nFiltered strategy (entities_to_keep={len(filtered_entities)}):")
print(f"  Only these entities count: {sorted(filtered_entities)}")

excluded_from_eval = all_model_entities - filtered_entities
print(f"\n  Model entities IGNORED ({len(excluded_from_eval)}):")
print(f"  {sorted(excluded_from_eval)}")
print("  → Predictions of these types won't count as FP or TP")

print("\nFull model strategy (entities_to_keep=None):")
print(f"  All {len(all_model_entities)} model entities are evaluated")
print("  → Model might detect entities not in dataset → could increase FP")

=== Evaluation Strategy Comparison ===

Filtered strategy (entities_to_keep=14):
  Only these entities count: ['CREDIT_CARD', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE', 'IP_ADDRESS', 'LOCATION', 'NRP', 'ORGANIZATION', 'PERSON', 'PHONE_NUMBER', 'TITLE', 'URL', 'US_DRIVER_LICENSE', 'US_SSN']

  Model entities IGNORED (7):
  ['CRYPTO', 'MAC_ADDRESS', 'MEDICAL_LICENSE', 'UK_NHS', 'US_BANK_NUMBER', 'US_ITIN', 'US_PASSPORT']
  → Predictions of these types won't count as FP or TP

Full model strategy (entities_to_keep=None):
  All 19 model entities are evaluated
  → Model might detect entities not in dataset → could increase FP


In [ ]:
wrapped_analyzer = PresidioAnalyzerWrapper(analyzer_engine=analyzer, language="en")

# Strategy 1: Filtered entities (recommended — avoids spurious FPs)
evaluator_filtered = SpanEvaluator(
    model=wrapped_analyzer,
    entities_to_keep=list(model_entities_to_use),
)

# Strategy 2: All model entities
evaluator_full = SpanEvaluator(
    model=wrapped_analyzer,
    entities_to_keep=None,
)

print("Ready to evaluate. Run the next cell to compare strategies.")

In [ ]:
%%time

subset = dataset[:100]

# Step 1: Predict
results_df_filtered = wrapped_analyzer.predict_dataset(subset)
results_df_full = wrapped_analyzer.predict_dataset(subset)

# Step 2: Map
mapped_filtered = mapper.get_mapped_results_dataframe(results_df_filtered)
mapped_full = mapper.get_mapped_results_dataframe(results_df_full)

# Step 3: Score
scores_filtered = evaluator_filtered.calculate_score_on_df(per_type=True, results_df=mapped_filtered)
scores_full = evaluator_full.calculate_score_on_df(per_type=True, results_df=mapped_full)

print("\n=== Strategy Comparison (100 samples) ===")
print(f"\n{'Metric':<20} {'Filtered':>12} {'Full Model':>12}")
print("-" * 44)
print(
    f"{'PII Precision':<20} {scores_filtered.pii_precision:>12.3f} {scores_full.pii_precision:>12.3f}"
)
print(
    f"{'PII Recall':<20} {scores_filtered.pii_recall:>12.3f} {scores_full.pii_recall:>12.3f}"
)
print(f"{'PII F1':<20} {scores_filtered.pii_f:>12.3f} {scores_full.pii_f:>12.3f}")

### Excluding Specific Model Entities

To prevent false positives from model entity types that have no counterpart in
your dataset, build the `entities_to_keep` set from the mapping values — as done
above. Any model entity not in that set is silently ignored during scoring.

In [ ]:
all_model_entities = set(analyzer.get_supported_entities("en"))
mapped_model_entities = {v for v in entity_mapping.values() if v is not None}
unused_model_entities = all_model_entities - mapped_model_entities

print(f"Model entities not in dataset mapping ({len(unused_model_entities)}):")
pprint(sorted(unused_model_entities), compact=True)
print(
    "\nPass `entities_to_keep=mapped_model_entities` to SpanEvaluator to ignore these."
)

Model entities not used by dataset (7):
['CRYPTO', 'MAC_ADDRESS', 'MEDICAL_LICENSE', 'UK_NHS', 'US_BANK_NUMBER',
 'US_ITIN', 'US_PASSPORT']
✓ Excluded 7 model entity(ies)


## Summary

### Key Takeaways

1. **Entity mapping is essential** for accurate evaluation when dataset and model use different entity label names.

2. **`CanonicalMapper`** auto-resolves labels through four tiers:
   - `EXACT` → known alias in the canonical taxonomy
   - `COUNTRY` → country-prefixed pattern (e.g. `GERMANY_PASSPORT_NUMBER` → `PASSPORT`)
   - `FUZZY` → high-similarity string match
   - `PENDING` → requires manual resolution via `.map()` or `.resolve_interactively()`

3. **Mapping to `None`** keeps the entity in evaluation but makes every instance a False Negative — use this when you want the model penalised for not detecting it.

4. **`.get_mapping()`** returns the final `{raw_label: canonical | None}` dict for `SpanEvaluator`. Raises `IncompleteMapping` if any labels are still pending.

5. **`entities_to_keep`** in `SpanEvaluator` controls which model prediction types matter — passing only the mapped canonical entities avoids spurious FPs from model-side entities absent in your dataset.

### Best Practices

1. Always call `mapper.render_html()` to review auto-resolved mappings before running experiments
2. Explicitly handle all pending labels (`.map()` or suppress to `None`)
3. Document your mapping decisions — the mapping dict is serialisable to JSON for experiment tracking

In [ ]:
print("""
=== CanonicalMapper Quick Reference ===

# Build from a dataset (InputSample list)
mapper = CanonicalMapper.from_dataset(dataset)
# Returns a dict directly when all labels resolve; otherwise returns a CanonicalMapper.

# Inspect auto-resolution
mapper.render_html()          # HTML audit table (Jupyter)
mapper._print_text()          # Plain-text fallback

# Check pending labels
mapper.pending                # list of unresolved labels

# Assign manually
mapper.map({
    "MY_LABEL": "PERSON",     # resolve to canonical entity
    "NOISE":    None,          # suppress from evaluation
})

# Interactive terminal resolution (shows ranked suggestions)
mapper.resolve_interactively()

# Get the final mapping dict (raises IncompleteMapping if pending is non-empty)
entity_mapping = mapper.get_mapping()  # {raw_label: canonical | None}

# New 3-step pipeline
results_df = model.predict_dataset(dataset)
mapped_df = mapper.get_mapped_results_dataframe(results_df)
scores = evaluator.calculate_score_on_df(per_type=True, results_df=mapped_df)

# Use with SpanEvaluator (no entity_mapping parameter)
entities_to_keep = {v for v in entity_mapping.values() if v is not None}
evaluator = SpanEvaluator(
    model=wrapped_model,
    entities_to_keep=list(entities_to_keep),
)

# Customise the taxonomy
from presidio_evaluator.entity_mapping import EntityHierarchy
h = EntityHierarchy.default().copy()
h.add_alias("EMAIL_ADDRESS", "E_MAIL")
mapper = CanonicalMapper.from_dataset(dataset, hierarchy=h)
""")